# 05 Open-Meteo-API und Kafka-Produzent

## Zweck
Rufen Sie aktuelle Open-Meteo-Luftqualitätsdaten für die ausgewählten europäischen Städte ab, bewahren Sie rohe JSON in Bronze auf, erstellen Sie versionierte Ereignisse und senden Sie sie optional an ein gruppenspezifisches Kafka-Thema.

## Eingaben
- `data/silver/city_reference.parquet`
- Open-Meteo-Luftqualität REST API
– `.env`-Werte für den optionalen Kafka-Produzenten

## Ausgaben
- `data/bronze/open_meteo_raw/<city_id>.json`
- `data/bronze/open_meteo_raw/open_meteo_air_quality_events.jsonl`
- `data/bronze/open_meteo_raw/open_meteo_ingestion_manifest.json`
– Kafka-Ereignisse auf `KAFKA_TOPIC_AIR_QUALITY_LIVE`, wenn FH Kafka aktiviert ist
- `data/samples/open_meteo_phase5_events_sample.jsonl`
- Lokaler Mock-Broker JSONL für reproduzierbare Offline-Tests

## Verwendete Technologien
Python, Anfragen, kafka-python, JSON.

## Konfiguration
Legen Sie `RUN_OPEN_METEO_API_FETCH=false` fest, um vorhandene lokale Bronze-JSON wiederzuverwenden. Legen Sie `RUN_OPEN_METEO_KAFKA_PRODUCER=true` auf FH JupyterHub fest, um es in Kafka zu veröffentlichen. `KAFKA_MODE=auto` versucht FH Kafka und greift auf einen transparenten lokalen JSONL-Mock-Broker zurück, wenn dies zulässig ist. Bei sicheren lokalen Ausführungen mit deaktiviertem Produzentenmodus wird der Mock-Broker direkt verwendet.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))
CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
OPEN_METEO_RAW_DIR = DATA_DIR / "bronze" / "open_meteo_raw"
SAMPLES_DIR = DATA_DIR / "samples"
OPEN_METEO_RAW_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print({"project_root": str(PROJECT_ROOT), "data_dir": str(DATA_DIR), "city_reference_exists": CITY_REFERENCE_PATH.exists()})


## Implementierung
Das Ereignisschema ist absichtlich flach, damit Spark es mit einem expliziten `StructType` im Notebook `06` analysieren kann. Das Notebook materialisiert immer lokale JSONL-Beweise. Die Kafka-Veröffentlichung ist ein geschützter externer Integrationsschritt.

### Konfigurieren Sie REST und die Kafka-Ereignisgenerierung

Diese Zelle deklariert Dateipfade, Feature-Flags, den Kafka-Modus und die Einstellungen für die begrenzte Wiedergabe. Behauptungen verhindern eine versehentliche Veröffentlichung in Platzhalter-Broker- oder Themenwerten.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
import json
import pandas as pd
import requests

OPEN_METEO_BASE_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
EVENTS_PATH = OPEN_METEO_RAW_DIR / "open_meteo_air_quality_events.jsonl"
MANIFEST_PATH = OPEN_METEO_RAW_DIR / "open_meteo_ingestion_manifest.json"
SAMPLE_EVENTS_PATH = SAMPLES_DIR / "open_meteo_phase5_events_sample.jsonl"
MOCK_BROKER_PATH = OPEN_METEO_RAW_DIR / "mock_kafka_air_quality_live.jsonl"
RUN_OPEN_METEO_API_FETCH = os.getenv("RUN_OPEN_METEO_API_FETCH", "true").lower() == "true"
ALLOW_CONTROLLED_OPEN_METEO_FALLBACK = os.getenv("ALLOW_CONTROLLED_OPEN_METEO_FALLBACK", "true").lower() == "true"
RUN_OPEN_METEO_KAFKA_PRODUCER = os.getenv("RUN_OPEN_METEO_KAFKA_PRODUCER", "false").lower() == "true"
BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "<kafka-host>:9092")
TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "LIVE-bdeng_gXX_air_quality_live")
KAFKA_MODE = os.getenv("KAFKA_MODE", "auto").lower()
ALLOW_KAFKA_MOCK_FALLBACK = os.getenv("ALLOW_KAFKA_MOCK_FALLBACK", "true").lower() == "true"
KAFKA_CONSUMER_TIMEOUT_MS = int(os.getenv("KAFKA_CONSUMER_TIMEOUT_MS", "10000"))
KAFKA_CONSUMER_MAX_MESSAGES = int(os.getenv("KAFKA_CONSUMER_MAX_MESSAGES", "8"))
KAFKA_CONNECTION_TIMEOUT_SECONDS = float(os.getenv("KAFKA_CONNECTION_TIMEOUT_SECONDS", "5"))
KAFKA_OPERATION_TIMEOUT_MS = int(os.getenv("KAFKA_OPERATION_TIMEOUT_MS", "15000"))
OPEN_METEO_REQUEST_TIMEOUT_SECONDS = int(os.getenv("OPEN_METEO_REQUEST_TIMEOUT_SECONDS", "20"))
OPEN_METEO_MAX_HOURS_TO_SEND = int(os.getenv("OPEN_METEO_MAX_HOURS_TO_SEND", "24"))

assert KAFKA_MODE in {"auto", "kafka", "mock"}, \
    f"KAFKA_MODE muss 'auto', 'kafka' oder 'mock' sein, erhalten: '{KAFKA_MODE}'"
assert OPEN_METEO_MAX_HOURS_TO_SEND >= 1, \
    f"OPEN_METEO_MAX_HOURS_TO_SEND muss >= 1 sein, erhalten: {OPEN_METEO_MAX_HOURS_TO_SEND}"
assert KAFKA_CONNECTION_TIMEOUT_SECONDS > 0, "KAFKA_CONNECTION_TIMEOUT_SECONDS muss > 0 sein"
assert KAFKA_OPERATION_TIMEOUT_MS >= 1000, "KAFKA_OPERATION_TIMEOUT_MS muss mindestens 1000 ms sein"
if RUN_OPEN_METEO_KAFKA_PRODUCER and KAFKA_MODE != "mock":
    assert "<" not in BOOTSTRAP_SERVERS, (
        f"KAFKA_BOOTSTRAP_SERVERS enthält einen Platzhalter ('{BOOTSTRAP_SERVERS}'). "
        "Set a real broker address in .env before enabling the Kafka producer."
    )
    assert "gXX" not in TOPIC, (
        f"KAFKA_TOPIC_AIR_QUALITY_LIVE enthält einen Platzhalter ('{TOPIC}'). "
        "Set your group-specific topic name in .env before enabling the Kafka producer."
    )

city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
required_city_columns = {"city_id", "city_name", "country_code", "latitude", "longitude"}
missing_city_cols = required_city_columns - set(city_reference_df.columns)
assert not missing_city_cols, f"In city_reference fehlen erforderliche Spalten: {missing_city_cols}"
assert city_reference_df["city_id"].is_unique, \
    f"Doppelte city_id-Werte in city_reference: {city_reference_df[city_reference_df['city_id'].duplicated()]['city_id'].tolist()}"

print({"kafka_mode": KAFKA_MODE, "run_producer": RUN_OPEN_METEO_KAFKA_PRODUCER, "topic": TOPIC, "cities": len(city_reference_df), "max_hours_to_send": OPEN_METEO_MAX_HOURS_TO_SEND, "connection_timeout_seconds": KAFKA_CONNECTION_TIMEOUT_SECONDS, "operation_timeout_ms": KAFKA_OPERATION_TIMEOUT_MS})


### Open-Meteo-Anfrageparameter erstellen

Die REST-Anfrage ist bewusst klein gehalten: ein UTC-Vorhersagetag und die drei im gesamten Projekt verwendeten Schadstoffe.

In [ ]:
def build_open_meteo_params(latitude: float, longitude: float) -> dict:
    return {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "pm2_5,pm10,nitrogen_dioxide",
        "timezone": "UTC",
        "forecast_days": 1,
    }

print("OK: build_open_meteo_params definiert")


### Definieren Sie deterministische Offline-API-Nutzlasten

Wenn dies ausdrücklich erlaubt ist, erstellt dieser Helfer vorhersehbare Fallback-Messungen. Ihr `controlled_fallback`-Marker verhindert analytischen Missbrauch.

In [ ]:
def controlled_open_meteo_payload(city_index: int) -> dict:
    times = [f"2026-01-15T{hour:02d}:00" for hour in range(24)]
    return {
        "hourly": {
            "time": times,
            "pm2_5": [float(6 + city_index + hour % 5) for hour in range(24)],
            "pm10": [float(12 + city_index + hour % 7) for hour in range(24)],
            "nitrogen_dioxide": [float(18 + city_index + hour % 9) for hour in range(24)],
        },
        "controlled_fallback": True,
    }

print("OK: controlled_open_meteo_payload definiert")


### Laden Sie die Open-Meteo-Nutzlast einer Stadt

Der Loader versucht zuerst API, dann den lokalen Bronze-JSON und dann den kontrollierten Fallback. Es gibt Herkunfts- und Abruffehler zusammen mit der Nutzlast zurück.

In [ ]:
def load_open_meteo_payload(city_row, city_index: int) -> tuple:
    raw_path = OPEN_METEO_RAW_DIR / f"{city_row['city_id']}.json"
    if RUN_OPEN_METEO_API_FETCH:
        try:
            response = requests.get(
                OPEN_METEO_BASE_URL,
                params=build_open_meteo_params(city_row["latitude"], city_row["longitude"]),
                timeout=OPEN_METEO_REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            payload = response.json()
            raw_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
            return payload, "fetched_api", None
        except requests.RequestException as exc:
            fetch_error = str(exc)
            if raw_path.exists():
                return json.loads(raw_path.read_text(encoding="utf-8")), "loaded_local_bronze_after_api_error", fetch_error
            if ALLOW_CONTROLLED_OPEN_METEO_FALLBACK:
                payload = controlled_open_meteo_payload(city_index)
                raw_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
                return payload, "controlled_offline_fallback", fetch_error
            raise
    if raw_path.exists():
        return json.loads(raw_path.read_text(encoding="utf-8")), "loaded_local_bronze", None
    if ALLOW_CONTROLLED_OPEN_METEO_FALLBACK:
        payload = controlled_open_meteo_payload(city_index)
        raw_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        return payload, "controlled_offline_fallback", "API-Abruf deaktiviert und keine lokale Bronze-JSON-Datei vorhanden"
    raise FileNotFoundError(f"Lokale Bronze-JSON-Datei fehlt: {raw_path}. RUN_OPEN_METEO_API_FETCH aktivieren.")

print("OK: load_open_meteo_payload definiert")


### Flachen Kafka-Ereignisvertrag aufbauen

Jedes Ereignis verwendet eine deterministische Hash-ID und ein flaches Schema. Dies macht das nachgelagerte Spark-Parsing und die mindestens einmalige Deduplizierung unkompliziert.

In [ ]:
def build_air_quality_event(city_id: str, timestamp_utc: str, hourly_row: dict, ingestion_time_utc: str, data_status: str) -> dict:
    event_key = f"{city_id}|{timestamp_utc}|open_meteo|1.0"
    return {
        "event_id": sha256(event_key.encode("utf-8")).hexdigest(),
        "schema_version": "1.0",
        "source": "open_meteo",
        "city_id": city_id,
        "event_time_utc": timestamp_utc,
        "ingestion_time_utc": ingestion_time_utc,
        "data_status": data_status,
        "pm2_5": hourly_row.get("pm2_5"),
        "pm10": hourly_row.get("pm10"),
        "no2": hourly_row.get("nitrogen_dioxide"),
    }

print("OK: build_air_quality_event definiert (schema_version=1.0, event_id=SHA256)")


### Konvertieren Sie stündliche API-Arrays in begrenzte Ereignisse

Dieser Helfer validiert Array-Längen, wählt vollständige Schadstoffstunden aus und begrenzt die Wiedergabegröße durch `OPEN_METEO_MAX_HOURS_TO_SEND`.

In [ ]:
def payload_to_events(city_id: str, payload: dict, ingestion_time_utc: str, data_status: str) -> list:
    hourly = payload.get("hourly", {})
    required_fields = ["time", "pm2_5", "pm10", "nitrogen_dioxide"]
    missing = [field for field in required_fields if field not in hourly]
    if missing:
        raise ValueError(f"Open-Meteo-Nutzlast für {city_id} enthält nicht alle Stundenfelder: {missing}")
    lengths = {field: len(hourly[field]) for field in required_fields}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Open-Meteo-Stundenarrays für {city_id} besitzen inkonsistente Längen: {lengths}")
    complete_indexes = [index for index in range(lengths["time"]) if all(hourly[field][index] is not None for field in required_fields if field != "time")]
    if not complete_indexes:
        raise ValueError(f"Open-Meteo-Nutzlast für {city_id} enthält keine vollständige Schadstoffstunde")
    selected_indexes = complete_indexes[-OPEN_METEO_MAX_HOURS_TO_SEND:]
    events = []
    for index in selected_indexes:
        timestamp = hourly["time"][index]
        hourly_row = {field: hourly[field][index] for field in required_fields if field != "time"}
        events.append(build_air_quality_event(city_id, f"{timestamp}:00Z", hourly_row, ingestion_time_utc, data_status))
    return events

print("OK: payload_to_events definiert")


### Alle Städte abrufen und Bronzebeweise aufbewahren

Die Schleife erstellt Ereignisse für jede Stadt, schreibt JSONL-Eingaben für Kafka und Spark und speichert ein Manifest, das API oder die Fallback-Herkunft beschreibt.

In [ ]:
ingestion_time_utc = datetime.now(timezone.utc).isoformat()
api_results = []
events = []
for city_index, (_, city_row) in enumerate(city_reference_df.iterrows()):
    payload, load_status, fetch_error = load_open_meteo_payload(city_row, city_index)
    data_status = "controlled_offline_fallback" if payload.get("controlled_fallback") else load_status
    city_events = payload_to_events(city_row["city_id"], payload, ingestion_time_utc, data_status)
    events.extend(city_events)
    raw_path = OPEN_METEO_RAW_DIR / f"{city_row['city_id']}.json"
    hourly = payload.get("hourly", {})
    missing_value_count = sum(value is None for field in ["pm2_5", "pm10", "nitrogen_dioxide"] for value in hourly.get(field, []))
    api_results.append({"city_id": city_row["city_id"], "city_name": city_row["city_name"], "status": data_status, "load_status": load_status, "http_status_code": 200 if load_status == "fetched_api" else None, "file_path": str(raw_path), "retrieved_at_utc": ingestion_time_utc, "event_count": len(city_events), "missing_value_count": missing_value_count, "fetch_error": fetch_error})

EVENTS_PATH.write_text("\n".join(json.dumps(event) for event in events) + "\n", encoding="utf-8")
events_df = pd.DataFrame(events)
api_results_df = pd.DataFrame(api_results)
MANIFEST_PATH.write_text(api_results_df.to_json(orient="records", indent=2), encoding="utf-8")
SAMPLE_EVENTS_PATH.write_text("\n".join(json.dumps(event) for event in events[:min(8, len(events))]) + "\n", encoding="utf-8")
api_results_df


## Validierung / Qualitätsprüfungen
Validieren Sie die API-Abdeckung, stabile Ereignisschlüssel, Zeitstempelform, zulässige Quellen, nicht negative Schadstoffwerte, sofern vorhanden, lokalen JSONL-Roundtrip, Kafka-Produzentenlieferung und begrenzte Verbrauchernachweise. Offline-Ausführungen verwenden den expliziten Mock-Broker.

### Generierte Ereignisse validieren

Vor der Veröffentlichung überprüft das Notebook Schema, Stadtabdeckung, deterministische IDs, Zeitstempel, Schadstoffbereiche und den lokalen JSONL-Roundtrip.

In [ ]:
required_event_columns = {
    "event_id", "schema_version", "source", "city_id", "event_time_utc",
    "ingestion_time_utc", "data_status", "pm2_5", "pm10", "no2",
}
missing_event_cols = required_event_columns - set(events_df.columns)
assert not missing_event_cols, f"In events_df fehlen erforderliche Spalten: {missing_event_cols}"

missing_cities = set(city_reference_df["city_id"]) - set(events_df["city_id"])
assert not missing_cities, f"Für diese Städte wurden keine Ereignisse erzeugt: {missing_cities}"

assert events_df["event_id"].is_unique, \
    f"Doppelte event_id-Werte gefunden: {events_df[events_df['event_id'].duplicated()]['event_id'].tolist()}"
assert (events_df["schema_version"] == "1.0").all(), \
    f"Unerwartete schema_version-Werte: {events_df['schema_version'].unique()}"
assert (events_df["source"] == "open_meteo").all(), \
    f"Unerwartete source-Werte: {events_df['source'].unique()}"

allowed_statuses = {"fetched_api", "loaded_local_bronze", "loaded_local_bronze_after_api_error", "controlled_offline_fallback"}
invalid_statuses = set(events_df["data_status"]) - allowed_statuses
assert not invalid_statuses, f"Unerwartete data_status-Werte: {invalid_statuses}"

assert events_df["event_time_utc"].str.endswith("Z").all(), \
    f"Einige event_time_utc-Werte enden nicht mit 'Z': {events_df.loc[~events_df['event_time_utc'].str.endswith('Z'), 'event_time_utc'].head().tolist()}"
assert events_df[["pm2_5", "pm10", "no2"]].apply(lambda values: values.dropna().ge(0).all()).all(), \
    "Negative Schadstoffwerte in Ereignissen gefunden"

roundtrip_events = [json.loads(line) for line in EVENTS_PATH.read_text(encoding="utf-8").splitlines()]
assert len(roundtrip_events) == len(events), \
    f"Abweichende Ereignisanzahl beim JSONL-Roundtrip: geschrieben: {len(events)}, erneut gelesen: {len(roundtrip_events)}"

print(f"OK: {len(events)} Ereignisse für {len(city_reference_df)} Städte validiert — event_id eindeutig, JSONL-Roundtrip bestanden")
print({"data_statuses": sorted(events_df['data_status'].unique()), "events_path": str(EVENTS_PATH)})


### Bereiten Sie die Smoke-Test-Identifikatoren des Brokers vor

Kafka-Verbrauchergruppen verwenden ein eindeutiges Suffix, sodass ein Rauchtest nur den aktuellen Lauf beobachten kann.

In [ ]:
from uuid import uuid4

print("OK: uuid4 importiert")


### Definieren Sie den lokalen Mock-Broker

Der Mock schreibt JSONL-Ereignisse und liest sie erneut. Es beweist die Mechanik der Produzenten/Konsumenten vor Ort, wird aber nie als FH-Kafka-Beweis vorgelegt.

In [ ]:
def publish_and_consume_mock(events: list) -> tuple:
    MOCK_BROKER_PATH.write_text("\n".join(json.dumps(event) for event in events) + "\n", encoding="utf-8")
    consumed = [json.loads(line) for line in MOCK_BROKER_PATH.read_text(encoding="utf-8").splitlines()[:KAFKA_CONSUMER_MAX_MESSAGES]]
    return {"mode": "mock", "topic": TOPIC, "sent": len(events), "delivery_errors": 0, "mock_path": str(MOCK_BROKER_PATH)}, consumed

print("OK: publish_and_consume_mock definiert")


### Definieren Sie den echten Kafka-Produzenten und den begrenzten Konsumenten

Der strikte Pfad startet mit einem kurzen TCP-Preflight. Danach erhalten auch Kafka-Metadatenabfragen, Zustellungen und Consumer-Abfragen explizite Zeitlimits. Fortschrittsausgaben zeigen, in welcher Netzwerkphase ein FH-Endpunkt scheitert. So bleibt der Nachweislauf diagnostizierbar und blockiert nicht unbegrenzt.


In [ ]:
def parse_bootstrap_servers(bootstrap_servers: str) -> list[tuple[str, int]]:
    endpoints = []
    for endpoint in bootstrap_servers.split(","):
        endpoint = endpoint.strip()
        if ":" not in endpoint:
            raise ValueError(f"Kafka-Endpunkt ohne Port: {endpoint!r}")
        host, port = endpoint.rsplit(":", 1)
        endpoints.append((host.strip(), int(port)))
    return endpoints

def tcp_preflight(bootstrap_servers: str) -> dict:
    import socket
    attempts = []
    for host, port in parse_bootstrap_servers(bootstrap_servers):
        try:
            with socket.create_connection((host, port), timeout=KAFKA_CONNECTION_TIMEOUT_SECONDS):
                attempts.append({"endpoint": f"{host}:{port}", "reachable": True, "reason": None})
        except Exception as exc:
            attempts.append({"endpoint": f"{host}:{port}", "reachable": False, "reason": str(exc)})
    if not any(attempt["reachable"] for attempt in attempts):
        raise ConnectionError(f"Kein Kafka-Bootstrap-Endpunkt per TCP erreichbar: {attempts}")
    return {"attempts": attempts}

def publish_and_consume_kafka(events: list) -> tuple:
    if TOPIC in {"air_quality_live", "bdeng_gXX_air_quality_live", "LIVE-bdeng_gXX_air_quality_live"} or "<" in BOOTSTRAP_SERVERS:
        raise ValueError("Vor der Veroeffentlichung einen erreichbaren Kafka-Broker und ein reales gruppenspezifisches Topic konfigurieren.")
    from time import monotonic
    from kafka import KafkaConsumer, KafkaProducer, TopicPartition

    print({"kafka_phase": "tcp_preflight", "bootstrap_servers": BOOTSTRAP_SERVERS, "timeout_seconds": KAFKA_CONNECTION_TIMEOUT_SECONDS}, flush=True)
    preflight = tcp_preflight(BOOTSTRAP_SERVERS)
    print({"kafka_phase": "tcp_preflight_ok", **preflight}, flush=True)

    producer = None
    consumer = None
    try:
        print({"kafka_phase": "producer_metadata", "topic": TOPIC, "timeout_ms": KAFKA_OPERATION_TIMEOUT_MS}, flush=True)
        producer = KafkaProducer(
            bootstrap_servers=BOOTSTRAP_SERVERS,
            key_serializer=lambda value: value.encode("utf-8"),
            value_serializer=lambda value: json.dumps(value).encode("utf-8"),
            api_version_auto_timeout_ms=KAFKA_OPERATION_TIMEOUT_MS,
            request_timeout_ms=KAFKA_OPERATION_TIMEOUT_MS,
            max_block_ms=KAFKA_OPERATION_TIMEOUT_MS,
        )
        partitions = producer.partitions_for(TOPIC)
        if not partitions:
            raise RuntimeError("Fuer das Kafka-Topic konnten keine Partitionen ermittelt werden. Topic und Broker-Konfiguration pruefen.")
        topic_partitions = [TopicPartition(TOPIC, partition) for partition in sorted(partitions)]

        print({"kafka_phase": "consumer_offsets", "partitions": sorted(partitions), "timeout_ms": KAFKA_OPERATION_TIMEOUT_MS}, flush=True)
        consumer = KafkaConsumer(
            bootstrap_servers=BOOTSTRAP_SERVERS,
            enable_auto_commit=False,
            value_deserializer=lambda value: json.loads(value.decode("utf-8")),
            api_version_auto_timeout_ms=KAFKA_OPERATION_TIMEOUT_MS,
            request_timeout_ms=KAFKA_OPERATION_TIMEOUT_MS,
            consumer_timeout_ms=KAFKA_CONSUMER_TIMEOUT_MS,
        )
        consumer.assign(topic_partitions)
        consumer.seek_to_end(*topic_partitions)
        start_offsets = {partition: consumer.position(partition) for partition in topic_partitions}

        print({"kafka_phase": "producer_send", "event_count": len(events)}, flush=True)
        futures = [producer.send(TOPIC, key=event["city_id"], value=event) for event in events]
        operation_timeout_seconds = KAFKA_OPERATION_TIMEOUT_MS / 1000
        for future in futures:
            future.get(timeout=operation_timeout_seconds)
        producer.flush(timeout=operation_timeout_seconds)

        expected_ids = {event["event_id"] for event in events}
        consumed = []
        deadline = monotonic() + KAFKA_CONSUMER_TIMEOUT_MS / 1000
        print({"kafka_phase": "consumer_poll", "timeout_ms": KAFKA_CONSUMER_TIMEOUT_MS}, flush=True)
        while monotonic() < deadline and len(consumed) < min(KAFKA_CONSUMER_MAX_MESSAGES, len(events)):
            remaining_ms = max(1, int((deadline - monotonic()) * 1000))
            records_by_partition = consumer.poll(timeout_ms=min(1000, remaining_ms), max_records=KAFKA_CONSUMER_MAX_MESSAGES)
            for records in records_by_partition.values():
                for message in records:
                    if message.value.get("event_id") in expected_ids:
                        consumed.append(message.value)
        if not consumed:
            raise RuntimeError("Der Kafka-Consumer-Smoke-Test hat kein Ereignis aus diesem Producer-Lauf empfangen.")
        print({"kafka_phase": "completed", "sent": len(events), "consumed": len(consumed)}, flush=True)
        return {"mode": "kafka", "topic": TOPIC, "sent": len(events), "delivery_errors": 0, "bootstrap_servers": BOOTSTRAP_SERVERS, "tcp_preflight": preflight, "start_offsets": {str(key): value for key, value in start_offsets.items()}}, consumed
    finally:
        if consumer is not None:
            consumer.close(autocommit=False)
        if producer is not None:
            producer.close(timeout=KAFKA_OPERATION_TIMEOUT_MS / 1000)

print("OK: publish_and_consume_kafka mit harten Zeitlimits definiert")


### Wählen Sie den Kafka- oder Mock-Modus

`KAFKA_MODE=auto` versucht es mit Kafka und greift nur dann zurück, wenn dies zulässig ist. `KAFKA_MODE=kafka` bleibt der strikte FH-Nachweis. Bei Netzwerkproblemen zeigen die Ausgaben die letzte erreichte Phase und der Lauf bricht mit einer konkreten Fehlermeldung ab.


In [ ]:
broker_result = None
consumed_events = []
fallback_reason = None
if RUN_OPEN_METEO_KAFKA_PRODUCER and KAFKA_MODE in {"auto", "kafka"}:
    try:
        broker_result, consumed_events = publish_and_consume_kafka(events)
    except Exception as exc:
        if KAFKA_MODE == "kafka" or not ALLOW_KAFKA_MOCK_FALLBACK:
            raise
        fallback_reason = str(exc)
        broker_result, consumed_events = publish_and_consume_mock(events)
else:
    broker_result, consumed_events = publish_and_consume_mock(events)

broker_result["consumed"] = len(consumed_events)
broker_result["fallback_reason"] = fallback_reason

print({"broker_mode": broker_result.get("mode"), "sent": broker_result.get("sent"), "consumed": len(consumed_events), "fallback_reason": fallback_reason})


### Validieren Sie die Beweise des Maklers

Die endgültigen Behauptungen überprüfen die Anzahl der Lieferungen und stellen sicher, dass mindestens ein verbrauchtes Ereignis dem Phase-5-Vertrag folgt.

In [ ]:
assert broker_result["sent"] == len(events), \
    f"Abweichende Broker-Sendeanzahl: erwartet {len(events)}, erhalten {broker_result['sent']}"
assert broker_result["delivery_errors"] == 0, \
    f"Broker meldete {broker_result['delivery_errors']} Zustellfehler"
assert len(consumed_events) >= 1, \
    "Consumer-Smoke-Test lieferte keine Ereignisse — Broker-Verbindung oder Mock-Pfad prüfen"
assert required_event_columns.issubset(consumed_events[0]), \
    f"Im konsumierten Ereignis fehlen Pflichtfelder: {required_event_columns - set(consumed_events[0])}"

print({"events_path": str(EVENTS_PATH), "manifest_path": str(MANIFEST_PATH), "event_count": len(events), "broker": broker_result})
print({"consumer_sample": consumed_events[0]})
events_df.head()


## Ergebnisse
Der Pfad REST API versucht alle ausgewählten Städte und erstellt einen lokalen Bronze-Ereignisstapel JSON sowie einen validierten JSONL-Ereignisbatch. Sollte der Netzwerkzugriff fehlschlagen, sorgen kontrollierte Fallback-Zeilen dafür, dass die Mechanik reproduzierbar bleibt und sichtbar gekennzeichnet bleibt. Mit `RUN_OPEN_METEO_KAFKA_PRODUCER=true` werden dieselben Ereignisse in Kafka veröffentlicht und durch einen begrenzten Consumer-Smoke-Test überprüft. Lokale Ausführungen verwenden den expliziten Mock-Broker.

## Einschränkungen
Live-API-Werte sind aktueller Kontext und dürfen nicht ohne Beschriftung mit historischen EEA-Schlussfolgerungen vermischt werden. Kontrollierte Fallback-Reihen und Mock-Broker-Zustellungen beweisen nur Mechanik und dürfen keine analytischen oder FH-Kafka-Behauptungen stützen. Die Kafka-Übermittlung verwendet die Semantik „mindestens einmal“. Die nachgelagerte Spark-Verarbeitung muss durch `event_id` dedupliziert werden. TCP-Erreichbarkeit allein beweist noch keinen korrekten Kafka advertised listener; deshalb begrenzen zusätzliche Client-Zeitlimits auch die Kafka-Metadatenphase.


## Nächster Schritt
Führen Sie das Notebook `06` für Spark Structured Streaming von Kafka zu Parquet aus.